# Step 3b · Features (part 2): flood extent and night lights

**Purpose**: add two kinds of satellite observation to the "exposed county × event" rows flagged in Step 3a. The two play different roles:

- **Sentinel-1 flood extent is a feature** (an explanatory variable): how much of the county was flooded. Step 4 uses it to test whether flooding adds anything beyond wind
- **Night lights are not a feature but a second observation of the outage**: what they see is the outage itself, so using them to predict outages would be predicting the answer from the answer. They serve only to cross-check EAGLE-I, and as a fallback outcome in counties where EAGLE-I has no record

A set of static variables that every county has is added as well: the share of land lying close to the level of the nearest stream. Counties whose radar image came late or not at all still have these.

**Structure**

| Part | Content | Data source | Needs Earth Engine | Approx. time |
| --- | --- | --- | --- | --- |
| A | Sample: exposed county × event rows | Step 3a results | No | 1 minute |
| B | Connect to Earth Engine; shared layers | GHSL, NLCD, JRC surface water, SRTM | Yes | 1 minute |
| C | Static flood susceptibility: share of land close to stream level | MERIT Hydro | Yes | 2–5 minutes |
| D | Sentinel-1 flood extent: image inventory, trial run, export tasks | Sentinel-1 GRD | Yes | About 5 minutes to submit; each task then runs on Google's servers for 1–5 minutes |
| E | Night lights: nightly extraction, conversion to percent of normal, comparison with EAGLE-I | VNP46A2 (Black Marble) | Yes | About 11 minutes |
| F | Read the flood results; flood features | Exports from part D | No | 1 minute |
| G | Merge; checklist | — | No | 1 minute |

Every part saves its result to the `us_flood_outage/step3b_cache` folder, so a rerun skips what is already done; flood results that have already been exported are not exported again.

**Running order**

1. An Earth Engine Cloud project is set as `PROJECT_ID` in cell 2, and the `us_flood_outage` folder holds the result files of Step 2c and Step 3a
2. Cell 8 submits the flood export tasks; the cells after it do not wait for them
3. Cell 13 reads the flood results that are finished, waiting at most `S1_WAIT_MIN` minutes; events that are not finished stay empty for now and show as "pending" in the checklist
4. Once all export tasks are done (progress at <https://code.earthengine.google.com/tasks>), cells 13–16 are run again

In [ ]:
# 1. Environment, output folder, chart style
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, TMP = "/content/drive/MyDrive/us_flood_outage", "/content"
except ImportError:                     # in local Jupyter: use the current folder
    OUT = "us_flood_outage" if os.path.isdir("us_flood_outage") else "../us_flood_outage"    # next to the notebook, or one level up
    TMP = f"{OUT}/tmp"
CACHE = f"{OUT}/step3b_cache"
for d in (OUT, TMP, CACHE):
    os.makedirs(d, exist_ok=True)
print("Output folder:", OUT)

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

def spearman(a, b):
    # Spearman rank correlation: rank first, then take the ordinary correlation
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank()) if ok.sum() >= 5 else np.nan

def cached(name, compute):
    # read the result from the cache folder if it is there, otherwise compute and save it
    path = f"{CACHE}/{name}.csv"
    if not os.path.exists(path):
        compute().to_csv(path, index=False)
    return pd.read_csv(path, dtype={"fips": str})

STATE_ABBR = {"01": "AL", "12": "FL", "13": "GA", "22": "LA", "28": "MS", "37": "NC", "45": "SC", "48": "TX"}
C_POWER, C_NTL = "#2a78d6", "#eb6834"
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## Parameters

The two flood thresholds and the night-light cleaning rules are the settings used on Florence in Step 1b. After changing a parameter, delete the `step3b_cache` folder and the `step3b_flood_*.csv` files in the output folder before rerunning; otherwise the old results are read again.

In [ ]:
# 2. Parameters
PROJECT_ID = "ee-yourname"       # Earth Engine Cloud project ID

# Sentinel-1 flood extent
S1_BEFORE_DAYS = (-36, -2)       # images before: 36 to 2 days before landfall, median
S1_AFTER_DAYS  = 12              # images after: within 12 days of landfall, darkest value
S1_TIMELY_DAYS = 6               # an image within this many days of landfall counts as timely
DIFF_DB, WATER_DB = -3.0, -15.0  # the two flood rules: more than 3 dB darker than before, and below −15 dB
SMOOTH_M, SLOPE_MAX, MIN_PIXELS = 50, 5, 8      # smoothing radius (m), maximum slope (degrees), minimum patch size (pixels)
S1_SCALE, S1_TEST_SCALE = 30, 90 # resolution (m) of the statistics; resolution of the trial run
S1_MIN_COVER = 50                # flood features are left empty when radar covers less than this share (%) of the county's land
S1_WAIT_MIN  = 10                # cell 13 waits at most this many minutes for the export tasks
S1_MAX_COUNTIES = 200            # events with more counties than this are exported as one task per state
RESUBMIT     = False             # True = resubmit export tasks that failed, or finished without a result file to be found
GHSL_YEAR, BUILT_MIN_M2 = 2015, 1000            # built-up: at least 1,000 m² of built surface in a 100 m cell (10%)

# Night lights
NTL_DAYS       = (-40, 21)       # extract every night from 40 days before landfall to 21 days after
NTL_BASE_END   = -2              # clear nights up to 2 days before landfall define "normal"
MIN_COVER      = 0.8             # a night is clear when at least 80% of the built-up pixels are usable
BUILT_MIN_FRAC = 0.10            # a 500 m pixel counts as built-up when at least 10% of it is built
NTL_MIN_PIXELS = 5               # no night-light values for counties with fewer built-up pixels than this
NTL_MIN_BASE   = 10              # viewing-angle adjustment needs at least this many clear nights before landfall; otherwise the median is used
NTL_CYCLE      = 16 / 3          # the satellite's viewing angle roughly repeats every 5.33 days
NTL_UTC_HOURS  = [6, 7, 8]       # hours (UTC) around the satellite overpass, used to read the EAGLE-I share with power
NTL_LOW_PCT    = 80              # "clearly dimmed" means below 80% of normal
NTL_BATCH, NTL_NIGHTS = 60, 21   # counties and nights per request

HAND_M   = (2, 5)                # land less than this many meters above the nearest stream counts as low
EE_BATCH = 40                    # counties per request elsewhere

## A. Sample

Read the Step 3a feature table and keep only the exposed counties (`exposed`). Flood extent and night lights are computed for these counties only.

`s1_day_step2c` is the "day of the first usable radar image after landfall" that Step 2c looked up at each county's center point; only affected counties have it. It is used below to check the area-based result of this step.

In [ ]:
# 3. Events and sample
EVENTS = pd.DataFrame([
    dict(key="harvey2017",   name="Harvey",   year=2017, landfall="2017-08-26 03:00"),
    dict(key="irma2017",     name="Irma",     year=2017, landfall="2017-09-10 13:00"),
    dict(key="florence2018", name="Florence", year=2018, landfall="2018-09-14 11:15"),
    dict(key="michael2018",  name="Michael",  year=2018, landfall="2018-10-10 17:30"),
    dict(key="laura2020",    name="Laura",    year=2020, landfall="2020-08-27 06:00"),
    dict(key="sally2020",    name="Sally",    year=2020, landfall="2020-09-16 09:45"),
    dict(key="zeta2020",     name="Zeta",     year=2020, landfall="2020-10-28 21:00"),
    dict(key="ida2021",      name="Ida",      year=2021, landfall="2021-08-29 16:55"),
    dict(key="ian2022",      name="Ian",      year=2022, landfall="2022-09-28 19:05"),
    dict(key="beryl2024",    name="Beryl",    year=2024, landfall="2024-07-08 08:40"),
    dict(key="helene2024",   name="Helene",   year=2024, landfall="2024-09-27 03:10"),
]).set_index("key")
EVENTS["landfall"] = pd.to_datetime(EVENTS["landfall"])

screen = pd.read_csv(f"{OUT}/step2c_event_screening.csv", index_col="event")
usable = [k for k in EVENTS.index if k in screen.index and bool(screen.loc[k, "usable"])]
EVENTS = EVENTS.loc[usable]

keep = ["event", "fips", "name", "state", "year", "customers_used", "peak_pct", "hours_above", "hours_rel", "affected", "long_outage",
        "no_record", "s1_first_day", "lat", "lon", "land_km2", "wind_swath_kt", "gust_mean_ms", "rain_max3d_mm"]
feat3a = pd.read_csv(f"{OUT}/step3a_features_county_event.csv", dtype={"fips": str})
sample = (feat3a[feat3a["exposed"] & feat3a["event"].isin(usable)][keep]
          .rename(columns={"s1_first_day": "s1_day_step2c"}).reset_index(drop=True))
FIPS_ALL = sorted(sample["fips"].unique())
def event_fips(key):
    return sorted(sample.loc[sample["event"] == key, "fips"])

print(f"{len(usable)} events, {len(sample):,} rows (exposed county × event), {len(FIPS_ALL)} counties; {int(sample['affected'].sum()):,} affected rows, {int(sample['long_outage'].sum()):,} with outages over 24 hours")
sample.groupby("event", sort=False).agg(counties=("fips", "size"), affected=("affected", "sum"), long_outage=("long_outage", "sum"),
                                        land_km2=("land_km2", "sum")).round(0)

## B. Connect to Earth Engine; shared layers

The cell below authenticates with Earth Engine.

The cell defines `county_sum`: the sum of an image's pixel values within each county, fetched in batches. When the request for a batch of counties fails, the batch is split in half and tried again. All shares below are obtained as "one sum divided by another" (for example flooded area ÷ land area), so a county with no data gets 0 rather than an empty value and is not dropped by Earth Engine.

Shared layers:

- **Built-up land** (GHSL 2015): at least 10% built surface in a 100 m cell
- **Land cover** (NLCD 2019): to tell wetlands, open water and cropland apart
- **Permanent water** (JRC): water present at least 10 months a year
- **Slope** (SRTM)

In [ ]:
# 4. [EE] Sign in to Earth Engine; fetch helpers; shared layers
import ee
ee.Authenticate()                         # follow the link to authorize
ee.Initialize(project=PROJECT_ID)
counties_fc = ee.FeatureCollection("TIGER/2018/Counties").filter(ee.Filter.inList("STATEFP", list(STATE_ABBR)))

def ee_rows(fc, props, tries=3):
    # fetch a few properties of a feature collection (Earth Engine drops features where any of them is empty)
    for attempt in range(tries):
        try:
            return fc.reduceColumns(ee.Reducer.toList(len(props)), props).get("list").getInfo()
        except Exception as err:
            if attempt == tries - 1:
                raise
            print(f"  request failed, retrying in {20 * (attempt + 1)} s:", str(err)[:150])
            time.sleep(20 * (attempt + 1))

def county_sum(img, bands, fips, batch, **grid):
    # for each band of the image, the sum of pixel values within each county. grid is scale=… or crs=…
    props = bands if len(bands) > 1 else ["sum"]                 # with a single band, Earth Engine names the result "sum"
    def fetch(part):
        fc = counties_fc.filter(ee.Filter.inList("GEOID", part))
        stats = img.select(bands).reduceRegions(collection=fc, reducer=ee.Reducer.sum(), tileScale=4, **grid)
        try:
            return ee_rows(stats, ["GEOID"] + props, tries=1 if len(part) > 1 else 3)
        except Exception as err:
            if len(part) == 1:
                raise
            print(f"  the request for {len(part)} counties did not succeed; splitting it in two:", str(err)[:120])
            half = len(part) // 2
            return fetch(part[:half]) + fetch(part[half:])
    rows = []
    for i in range(0, len(fips), batch):
        rows += fetch(fips[i:i + batch])
    return pd.DataFrame(rows, columns=["fips"] + bands).set_index("fips").reindex(fips).reset_index()

ghsl = (ee.ImageCollection("JRC/GHSL/P2023A/GHS_BUILT_S")
          .filterDate(f"{GHSL_YEAR}-01-01", f"{GHSL_YEAR + 1}-01-01").first().select("built_surface"))
builtup = ghsl.gte(BUILT_MIN_M2).unmask(0)                                       # built-up land
lc = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").filter(ee.Filter.eq("system:index", "2019")).first().select("landcover")
not_open_water = lc.neq(11).unmask(0)                                            # 11 = open water
dry  = lc.neq(11).And(lc.neq(90)).And(lc.neq(95)).unmask(0)                      # neither open water nor wetland (90, 95)
crop = lc.eq(81).Or(lc.eq(82)).unmask(0)                                         # pasture and cultivated crops
perm_water = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("seasonality").gte(10).unmask(0)
slope = ee.Terrain.slope(ee.Image("USGS/SRTMGL1_003"))
print("Earth Engine initialized")

## C. Static flood susceptibility: share of land close to stream level

MERIT Hydro gives, for each 90 m pixel, how many meters it lies above the nearest stream it drains to (height above nearest drainage, HAND). The smaller this height, the more easily the pixel floods when the stream overflows.

| Column | Meaning |
| --- | --- |
| `hand_le2m_pct`, `hand_le5m_pct` | Share of the county's land less than 2 m and 5 m above the nearest stream |
| `built_hand_le2m_pct`, `built_hand_le5m_pct` | The same share within built-up land. Empty for counties with no built-up land |

This variable does not change from event to event, exists for every county, and does not depend on when the radar passed over. Its limitation is that it reflects susceptibility to river flooding only, not to storm surge.

In [ ]:
# 5. [EE] Share of land close to stream level
hnd = ee.Image("MERIT/Hydro/v1_0_1").select("hnd")

def hand_static():
    layers = {"land_n": not_open_water, "built_n": builtup}
    for m in HAND_M:
        low = hnd.lte(m).unmask(0)
        layers[f"land_low{m}"]  = low.And(not_open_water)
        layers[f"built_low{m}"] = low.And(builtup)
    img = ee.Image.cat([v.rename(k) for k, v in layers.items()])
    return county_sum(img, list(layers), FIPS_ALL, EE_BATCH, scale=90)

hand = cached("hand_static", hand_static)
HAND_COLS = []
for m in HAND_M:
    hand[f"hand_le{m}m_pct"]       = hand[f"land_low{m}"]  / hand["land_n"].where(hand["land_n"] > 0) * 100
    hand[f"built_hand_le{m}m_pct"] = hand[f"built_low{m}"] / hand["built_n"].where(hand["built_n"] > 0) * 100
    HAND_COLS += [f"hand_le{m}m_pct", f"built_hand_le{m}m_pct"]
print(f"{int(hand[HAND_COLS[0]].notna().sum())} / {len(FIPS_ALL)} counties with a value")
tidy(hand[HAND_COLS].describe().loc[["min", "50%", "max"]])

## D. Sentinel-1 flood extent

**Method** (the same as in Step 1b): calm water reflects the radar signal away, so flooded areas turn dark in radar images. A pixel counts as flooded when both conditions hold:

1. It is more than 3 dB darker after than before
2. Its post-event value is below −15 dB

Permanent water, slopes above 5° and patches smaller than 8 pixels are then removed. Before and after are compared only within the **same orbit**. "Before" is the median from 36 to 2 days before landfall; "after" is the darkest value within 12 days of landfall.

**Three differences from Step 1b**

- **The day of the first post-landfall image is recorded for each pixel**. Floods recede, so a late image sees less. `s1_day` is the area-weighted mean day for the county; 6 days or less counts as timely
- **The flooded share is also reported without wetlands** (`flood_dry_pct`). Coastal marsh is wet and dry by turns anyway; with wetlands included, the coastal parishes of Louisiana would show large numbers that have little to do with power infrastructure
- **The share of cropland in the flooded area is reported** (`flood_crop_share`), along with a flooded share that leaves out both wetlands and cropland (`flood_nocrop_pct`). Harvested fields and crops flattened by wind also turn dark. In the first run most of the counties with the highest flooded share were of this kind, so the figure and tables below go by `flood_nocrop_pct`

| Column | Meaning |
| --- | --- |
| `s1_cover_pct` | Share of the county's land covered by radar (images both before and after) |
| `s1_day` | Day of the first usable image after landfall (area-weighted mean) |
| `s1_timely_pct`, `s1_timely` | Share of land imaged within 6 days; timely when more than half |
| `flood_pct` | Share of land flagged as flooded |
| `flood_dry_pct` | The same, wetlands excluded |
| `flood_nocrop_pct` | The same, wetlands and cropland both excluded |
| `flood_built_pct` | Share of built-up land flagged as flooded |
| `flood_crop_share` | Share of cropland in the flooded area |

**Known limitations**: floods in built-up areas and under forest are missed (strong reflections from buildings and trunks swamp the water signal); in Step 1b even the worst-flooded Florence county had only 1.4% of its land flagged. So these numbers suit comparisons of "which county flooded relatively more", not statements about true flooded area. Also, if another flood happened within the "before" window (Florence three weeks before Michael, for example), the "before" images are themselves dark and the flood is underestimated.

The next three cells do three things in turn: find the usable orbits of each event; run a trial on one county per event to confirm that the flood map can be computed; submit the export tasks. Events with more than `S1_MAX_COUNTIES` counties are split into one task per state.

In [ ]:
# 6. [EE] Sentinel-1 image inventory and usable orbits for each event
def iso(t):
    return t.strftime("%Y-%m-%dT%H:%M:%S")

def s1_windows(key):
    # the two time windows, before and after (UTC)
    lf = EVENTS.loc[key, "landfall"]
    return ((iso(lf + pd.Timedelta(days=S1_BEFORE_DAYS[0])), iso(lf + pd.Timedelta(days=S1_BEFORE_DAYS[1]))),
            (iso(lf), iso(lf + pd.Timedelta(days=S1_AFTER_DAYS))))

def s1_collection(key):
    # Sentinel-1 images (VV polarization) over the exposed counties of this event
    g = sample[sample["event"] == key]
    box = [float(g["lon"].min()) - 0.8, float(g["lat"].min()) - 0.8, float(g["lon"].max()) + 0.8, float(g["lat"].max()) + 0.8]
    return (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(ee.Geometry.Rectangle(box))
              .filter(ee.Filter.eq("instrumentMode", "IW"))
              .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
              .select("VV")
              .map(lambda img: img.updateMask(img.gt(-30))))             # remove noise at the image edges

def s1_inventory(key):
    col, rows = s1_collection(key), []
    for period, (t0, t1) in zip(["before", "after"], s1_windows(key)):
        found = ee_rows(col.filterDate(t0, t1), ["system:time_start", "orbitProperties_pass", "relativeOrbitNumber_start"])
        rows += [r + [period] for r in found]
    return pd.DataFrame(rows, columns=["t", "pass", "orbit", "period"])

usable_orbits, inv_rows = {}, []
for key in usable:
    inv = cached(f"s1_inventory_{key}", lambda: s1_inventory(key))
    inv["orbit"] = inv["orbit"].astype(int)
    inv["day"] = (pd.to_datetime(inv["t"], unit="ms") - EVENTS.loc[key, "landfall"]) / pd.Timedelta(days=1)
    b, a = inv[inv["period"] == "before"], inv[inv["period"] == "after"]
    usable_orbits[key] = sorted(int(r) for r in set(b["orbit"]) & set(a["orbit"]))     # orbits with images both before and after
    ok = a[a["orbit"].isin(usable_orbits[key])]
    inv_rows.append({"event": key, "images_before": len(b), "images_after": len(a), "orbits_before": b["orbit"].nunique(),
                     "orbits_after": a["orbit"].nunique(), "orbits_usable": len(usable_orbits[key]),
                     "first_usable_day": ok["day"].min(), "last_usable_day": ok["day"].max()})
s1_tab = pd.DataFrame(inv_rows).set_index("event")
tidy(s1_tab)

In [ ]:
# 7. [EE] Definition of the flood map and the county statistics; a trial on one county per event
S1_BANDS = ["land_km2", "flood_km2", "dry_km2", "flood_dry_km2", "built_km2", "flood_built_km2", "crop_km2", "flood_crop_km2",
            "timely_km2", "day_km2"]

def flood_stats_image(key):
    # an image with 10 bands, each "km² of some category in each pixel"; summing within a county gives areas
    lf = EVENTS.loc[key, "landfall"]
    lf_ms = int((lf - pd.Timestamp("1970-01-01")) / pd.Timedelta(milliseconds=1))
    (b0, b1), (a0, a1) = s1_windows(key)
    col = s1_collection(key)
    before, after = col.filterDate(b0, b1), col.filterDate(a0, a1)

    def one_orbit(r):
        same  = ee.Filter.eq("relativeOrbitNumber_start", r)
        b     = before.filter(same).median().focalMedian(SMOOTH_M, "circle", "meters")
        a_col = after.filter(same)
        a     = a_col.min().focalMedian(SMOOTH_M, "circle", "meters")
        wet   = a.subtract(b).lt(DIFF_DB).And(a.lt(WATER_DB)).rename("flood")
        # turn each post-landfall image into "days since landfall" and keep the earliest
        day   = (a_col.map(lambda im: im.multiply(0).add(ee.Number(im.get("system:time_start")).subtract(lf_ms).divide(86400000)).rename("day"))
                      .min().updateMask(wet.mask()))
        return wet.addBands(day)

    per_orbit  = ee.ImageCollection([one_orbit(r) for r in usable_orbits[key]])
    flood_raw  = per_orbit.select("flood").max()                        # flooded if any orbit flags it
    first_day  = per_orbit.select("day").min()
    candidate  = flood_raw.And(perm_water.Not()).And(slope.lt(SLOPE_MAX))
    big_enough = candidate.selfMask().connectedPixelCount(MIN_PIXELS + 2, True).gte(MIN_PIXELS).unmask(0)
    flood      = candidate.And(big_enough)
    seen       = flood.gte(0).And(perm_water.Not()).unmask(0)           # land covered by the radar
    flooded    = flood.unmask(0)
    layers = {
        "land_km2": seen,                  "flood_km2": flooded,
        "dry_km2": seen.And(dry),          "flood_dry_km2": flooded.And(dry),
        "built_km2": seen.And(builtup),    "flood_built_km2": flooded.And(builtup),
        "crop_km2": seen.And(crop),        "flood_crop_km2": flooded.And(crop),
        "timely_km2": seen.And(first_day.lte(S1_TIMELY_DAYS).unmask(0)),
        "day_km2": first_day.unmask(0).multiply(seen),
    }
    km2 = ee.Image.pixelArea().divide(1e6)
    return ee.Image.cat([layers[name].multiply(km2).rename(name) for name in S1_BANDS])

def flood_file(name):
    return f"{OUT}/{name}.csv"

def flood_jobs(key):
    # the export tasks of this event (task name → counties). One task for the whole event if its result is already in Drive or it has few counties; otherwise one per state
    fips, whole = event_fips(key), f"step3b_flood_{key}"
    if os.path.exists(flood_file(whole)) or len(fips) <= S1_MAX_COUNTIES:
        return {whole: fips}
    return {f"{whole}_{STATE_ABBR[s]}": [f for f in fips if f[:2] == s] for s in sorted({f[:2] for f in fips})}

def flood_done(key):
    return all(os.path.exists(flood_file(name)) for name in flood_jobs(key))

trial = []
for key in usable:
    if flood_done(key) or not usable_orbits[key]:
        continue                                                        # result already there, or no usable orbit
    g = sample[sample["event"] == key].nlargest(8, "rain_max3d_mm").nsmallest(1, "land_km2").iloc[0]     # the smallest of the 8 rainiest counties
    try:
        fc = counties_fc.filter(ee.Filter.inList("GEOID", [g["fips"]]))
        stats = flood_stats_image(key).reduceRegions(collection=fc, reducer=ee.Reducer.sum(), scale=S1_TEST_SCALE, tileScale=4)
        v = dict(zip(S1_BANDS, ee_rows(stats, ["GEOID"] + S1_BANDS, tries=1)[0][1:]))
        trial.append({"event": key, "county": g["name"], "rain_max3d_mm": g["rain_max3d_mm"], "county_land_km2": g["land_km2"],
                      "radar_land_km2": v["land_km2"], "flood_pct": v["flood_km2"] / max(v["land_km2"], 1e-9) * 100,
                      "first_image_day": v["day_km2"] / max(v["land_km2"], 1e-9)})
    except Exception as err:
        print(f"{key}: the trial did not succeed; the export task is submitted anyway. Reason:", str(err)[:300])
trial = pd.DataFrame(trial)
print(f"Trial succeeded for {len(trial)} events (at {S1_TEST_SCALE} m, only to confirm the map computes; the real statistics use {S1_SCALE} m)")
tidy(trial, 2)

In [ ]:
# 8. [EE] Submit the export tasks for the flood statistics (they run on Google's servers; one task per state for events with many counties)
def task_states():
    # state of the latest submission under each task name
    latest = {}
    for t in ee.data.getTaskList():
        d = t.get("description")
        if d not in latest or t.get("creation_timestamp_ms", 0) > latest[d].get("creation_timestamp_ms", 0):
            latest[d] = t
    return latest

states = task_states()
for key in usable:
    for name, fips in flood_jobs(key).items():
        state = states.get(name, {}).get("state")
        if os.path.exists(flood_file(name)):
            print(f"{name}: result already in Drive")
        elif not usable_orbits[key]:
            print(f"{name}: no orbit with images both before and after; skipped")
        elif state in ("READY", "RUNNING"):
            print(f"{name}: task already running; not submitted again")
        elif state == "COMPLETED" and not RESUBMIT:
            print(f"{name}: task completed; waiting for the file to appear in Drive (if it never does, set RESUBMIT to True)")
        elif state in ("FAILED", "CANCELLED", "CANCEL_REQUESTED") and not RESUBMIT:
            print(f"{name}: the last task did not succeed. Reason:", str(states[name].get("error_message", state))[:300])
        else:
            fc = counties_fc.filter(ee.Filter.inList("GEOID", fips)).select(["GEOID", "ALAND"])
            stats = flood_stats_image(key).reduceRegions(collection=fc, reducer=ee.Reducer.sum(), scale=S1_SCALE, tileScale=4)
            ee.batch.Export.table.toDrive(collection=stats, description=name, folder="us_flood_outage", fileNamePrefix=name,
                                          fileFormat="CSV", selectors=["GEOID", "ALAND"] + S1_BANDS).start()
            print(f"{name}: submitted ({len(fips)} counties, {len(usable_orbits[key])} orbits)")
print("Progress: https://code.earthengine.google.com/tasks")

## E. Night lights

**Cleaning** (the same as in Step 1b): keep only high-quality, confidently clear pixels with no cloud shadow or thin cirrus; look only at built-up land; a night counts as clear only when at least 80% of the built-up pixels are usable.

**Conversion to "percent of normal"**: the satellite looks at the same city from a different angle each night, the angle roughly repeats every 5.33 days, and the same lights can look 20% brighter or dimmer from one angle to another. So the notebook first fits how normal radiance cycles with viewing angle on the clear nights before landfall, then compares each night with the normal radiance for its angle. Counties with fewer than 10 clear nights before landfall skip this step and are compared with the median; counties with fewer than 3 get no value.

| Column | Meaning |
| --- | --- |
| `ntl_pixels` | Number of built-up pixels in the county |
| `ntl_base_nights` | Number of clear nights before landfall |
| `ntl_noise_pct` | Normal night-to-night variation (after the viewing-angle adjustment, %). A dimming has to be clearly larger than this to count |
| `ntl_first_day`, `ntl_first_pct` | Day of the first clear night after landfall; that night's radiance as a percentage of normal |
| `ntl_nights_7d` | Number of clear nights within 7 days of landfall |
| `ntl_min_pct_7d`, `ntl_mean_pct_7d` | Lowest and mean radiance on clear nights within 7 days of landfall (normal = 100) |
| `ntl_mean_pct_8_14d` | Mean radiance on clear nights 8–14 days after landfall |

**Comparison with EAGLE-I**: a night-light image dated on a given day corresponds to 6–9 UTC of that day (1–3 AM local time). Affected counties have the hourly outage curves saved by Step 2c; the mean of those three hours is converted to "share of customers with power" and compared with the night lights of the same night.

**Known limitations**: hurricanes are often followed by days of cloud, the first clear night commonly comes three to five days after landfall, and night lights cannot see the worst days of the outage; thin cloud sometimes slips through the screening and causes false dimming; and if another outage happened within the 40 days before landfall, "normal" is pulled down a little. The counties that were not affected are used below to estimate how common false dimming is.

In [ ]:
# 9. [EE] Extract the built-up radiance of each county on each night (the slowest cell)
NTL_COL, NTL_BAND = "NASA/VIIRS/002/VNP46A2", "DNB_BRDF_Corrected_NTL"
QF_BAND, CLOUD_BAND = "Mandatory_Quality_Flag", "QF_Cloud_Mask"

raw_ntl = ee.ImageCollection(NTL_COL)
proj = raw_ntl.first().select(NTL_BAND).projection()
# GHSL built-up surface (m² per 100 m cell) → built-up fraction of each 500 m pixel
built500   = ghsl.divide(10000).reduceResolution(ee.Reducer.mean(), maxPixels=64).reproject(proj)
settlement = built500.gte(BUILT_MIN_FRAC).unmask(0)

def clean(img):
    ok = img.select(QF_BAND).lte(1)                                   # 0/1 = high quality
    qf = img.select(CLOUD_BAND)
    clear  = qf.rightShift(6).bitwiseAnd(3).eq(0)                     # bits 6–7: keep "confident clear" only
    shadow = qf.rightShift(8).bitwiseAnd(1).eq(0)                     # bit 8: no cloud shadow
    cirrus = qf.rightShift(9).bitwiseAnd(1).eq(0)                     # bit 9: no thin cirrus
    return img.select(NTL_BAND).updateMask(ok.And(clear).And(shadow).And(cirrus)).rename("ntl")

def ntl_dates(key):
    # the dates with a night-light image within this event's window
    day0 = EVENTS.loc[key, "landfall"].normalize()
    t0, t1 = day0 + pd.Timedelta(days=NTL_DAYS[0]), day0 + pd.Timedelta(days=NTL_DAYS[1] + 1)
    ms = raw_ntl.filterDate(t0.strftime("%Y-%m-%d"), t1.strftime("%Y-%m-%d")).aggregate_array("system:time_start").getInfo()
    return pd.DataFrame({"date": sorted({pd.Timestamp(int(m), unit="ms").strftime("%Y-%m-%d") for m in ms})})

def ntl_chunk(key, dates):
    # two bands per night: summed radiance of the usable built-up pixels (s) and their number (c)
    layers, bands = [], []
    for d in dates:
        tag = d.replace("-", "")
        img = clean(raw_ntl.filterDate(d, (pd.Timestamp(d) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")).first())
        ok  = img.mask().And(settlement)
        layers += [img.unmask(0).multiply(ok).rename("s" + tag), ok.rename("c" + tag)]
        bands  += ["s" + tag, "c" + tag]
    wide = county_sum(ee.Image.cat(layers), bands, event_fips(key), NTL_BATCH, crs=proj)
    return pd.concat([pd.DataFrame({"fips": wide["fips"], "date": d, "sum": wide["s" + d.replace("-", "")],
                                    "count": wide["c" + d.replace("-", "")]}) for d in dates], ignore_index=True)

ntl_parts = []
for key in usable:
    t_start = time.time()
    dates = cached(f"ntl_dates_{key}", lambda: ntl_dates(key))["date"].tolist()
    for i in range(0, len(dates), NTL_NIGHTS):
        part = cached(f"ntl_{key}_{i // NTL_NIGHTS:02d}", lambda: ntl_chunk(key, dates[i:i + NTL_NIGHTS]))
        ntl_parts.append(part.assign(event=key))
    print(f"{key}: {len(event_fips(key))} counties × {len(dates)} nights, {(time.time() - t_start) / 60:.1f} minutes")
ntl_raw = pd.concat(ntl_parts, ignore_index=True)
pixels  = cached("ntl_pixels", lambda: county_sum(settlement.rename("n_elig"), ["n_elig"], FIPS_ALL, EE_BATCH, crs=proj))
print(f"Extraction done: {len(ntl_raw):,} rows (county × night)")

In [ ]:
# 10. Convert night lights to "percent of normal" and compute the night-light indicators of each county × event
OVERPASS_H = float(np.mean(NTL_UTC_HOURS)) + 0.5                      # overpass time (UTC hour)

nights = ntl_raw.copy()
nights["date"]   = pd.to_datetime(nights["date"])
nights = nights.sort_values(["event", "fips", "date"]).reset_index(drop=True)
nights["n_elig"] = nights["fips"].map(pixels.set_index("fips")["n_elig"])
nights["mean"]   = (nights["sum"] / nights["count"]).where(nights["count"] > 0)
nights["clear"]  = (nights["count"] >= MIN_COVER * nights["n_elig"]) & (nights["n_elig"] >= NTL_MIN_PIXELS) & (nights["mean"] > 0)
landfall = nights["event"].map(EVENTS["landfall"])
nights["day"] = (nights["date"] + pd.Timedelta(hours=OVERPASS_H) - landfall) / pd.Timedelta(days=1)     # days since landfall (fractional)
nights["k"]   = (nights["date"] - landfall.dt.normalize()).dt.days                                      # whole days from the landfall date, for the viewing-angle cycle

def angle_terms(k):
    # the viewing-angle cycle: a constant plus one cosine–sine pair
    phase = 2 * np.pi * np.asarray(k, dtype=float) / NTL_CYCLE
    return np.column_stack([np.ones(len(phase)), np.cos(phase), np.sin(phase)])

def expected_ntl(base_k, base_v, k):
    # fit how radiance varies with viewing angle on normal nights; return the radiance expected on nights k under normal conditions
    if len(base_v) < NTL_MIN_BASE:                                 # too few clear nights: fall back to the median
        return np.full(len(k), np.median(base_v))
    coef = np.linalg.lstsq(angle_terms(base_k), np.log(base_v), rcond=None)[0]
    return np.exp(angle_terms(k) @ coef)

rows, pct = [], pd.Series(np.nan, index=nights.index)
for (key, f), g in nights[nights["clear"]].groupby(["event", "fips"], sort=False):
    base = g[g["day"] <= NTL_BASE_END]
    rec = {"event": key, "fips": f, "ntl_base_nights": len(base)}
    if len(base) >= 3:
        bk, bv = base["k"].values, base["mean"].values
        p = g["mean"].values / expected_ntl(bk, bv, g["k"].values) * 100
        pct.loc[g.index] = p
        # noise: leave each pre-landfall clear night out in turn, predict it from the others, and see how far off it is
        left_out = [bv[i] / expected_ntl(np.delete(bk, i), np.delete(bv, i), bk[[i]])[0] for i in range(len(bv))]
        post = pd.Series(p, index=g["day"].values)
        post = post[post.index > 0]
        w1, w2 = post[post.index <= 7], post[(post.index > 7) & (post.index <= 14)]
        rec.update({
            "ntl_noise_pct": float(np.std(left_out, ddof=1) * 100),
            "ntl_first_day": float(post.index[0]) if len(post) else np.nan,
            "ntl_first_pct": float(post.iloc[0]) if len(post) else np.nan,
            "ntl_nights_7d": len(w1),
            "ntl_min_pct_7d": float(w1.min()) if len(w1) else np.nan,
            "ntl_mean_pct_7d": float(w1.mean()) if len(w1) else np.nan,
            "ntl_mean_pct_8_14d": float(w2.mean()) if len(w2) else np.nan,
        })
    rows.append(rec)
nights["ntl_pct"] = pct
NTL_COLS = ["ntl_pixels", "ntl_base_nights", "ntl_noise_pct", "ntl_first_day", "ntl_first_pct", "ntl_nights_7d", "ntl_min_pct_7d",
            "ntl_mean_pct_7d", "ntl_mean_pct_8_14d"]
ntl_feat = sample[["event", "fips"]].merge(pd.DataFrame(rows, columns=["event", "fips"] + NTL_COLS[1:]), on=["event", "fips"], how="left")
ntl_feat.insert(2, "ntl_pixels", ntl_feat["fips"].map(pixels.set_index("fips")["n_elig"]))
ntl_feat["ntl_base_nights"] = ntl_feat["ntl_base_nights"].fillna(0)                         # counties without a single clear night

# EAGLE-I: share of customers with power around the satellite overpass, for affected counties
hourly = pd.read_csv(f"{OUT}/step2c_panel_hourly.csv.gz", dtype={"fips": str}, parse_dates=["time"])
hourly = hourly[hourly["event"].isin(usable) & hourly["time"].dt.hour.isin(NTL_UTC_HOURS)]
out_night = (hourly.assign(date=hourly["time"].dt.normalize()).groupby(["event", "fips", "date"])["customers_out"].mean()
                   .rename("out_night").reset_index())
info = sample[["event", "fips", "customers_used", "affected", "hours_above"]]
nights = nights.merge(info, on=["event", "fips"], how="left").merge(out_night, on=["event", "fips", "date"], how="left")
nights["power_pct"] = ((1 - nights.pop("out_night") / nights["customers_used"]).clip(0, 1) * 100).where(nights["affected"])
nights[["event", "fips", "date", "day", "count", "n_elig", "mean", "clear", "ntl_pct", "power_pct"]].to_csv(
    f"{OUT}/step3b_ntl_nights.csv.gz", index=False, compression="gzip")

have = ntl_feat["ntl_base_nights"] >= 3
print(f"Of {len(ntl_feat):,} rows, {int(have.sum()):,} have a night-light baseline; median number of clear nights before landfall {ntl_feat.loc[have, 'ntl_base_nights'].median():.0f}, median normal variation {ntl_feat['ntl_noise_pct'].median():.1f}%")
print(f"{int((ntl_feat['ntl_nights_7d'] >= 1).sum()):,} rows have at least one clear night within 7 days of landfall; day of the first clear night after landfall (median): {ntl_feat['ntl_first_day'].median():.1f}")
tidy(ntl_feat[NTL_COLS].describe().loc[["min", "25%", "50%", "75%", "max"]])

In [ ]:
# 11. Do night lights agree with EAGLE-I: Figure 1 and the comparison table by event
post  = nights[nights["clear"] & nights["ntl_pct"].notna() & (nights["day"] > 0) & (nights["day"] <= 14)]
pairs = post[post["power_pct"].notna()].copy()                       # affected counties: nights with both night lights and outage data
quiet = post[~post["affected"]]                                      # exposed but not affected counties: night lights should hardly change
BIN_NAMES = ["Under 50%", "50–80%", "80–95%", "95–100%"]
pairs["bin"] = pd.cut(pairs["power_pct"], [-0.01, 50, 80, 95, 100], labels=BIN_NAMES)

def box(g):
    return {"nights": len(g), "counties": len(g[["event", "fips"]].drop_duplicates()),
            "ntl_q25": g["ntl_pct"].quantile(0.25), "ntl_median": g["ntl_pct"].median(), "ntl_q75": g["ntl_pct"].quantile(0.75),
            "dim_pct": (g["ntl_pct"] < NTL_LOW_PCT).mean() * 100}
ntl_bins = pd.DataFrame({**{b: box(pairs[pairs["bin"] == b]) for b in BIN_NAMES}, "Not affected": box(quiet)}).T
ntl_bins.index.name = "customers_with_power"
ntl_bins.to_csv(f"{OUT}/step3b_ntl_by_power_bin.csv")
rho_nights = spearman(pairs["ntl_pct"], pairs["power_pct"])

def darker(a, b):
    # probability that a random value from a is smaller than a random value from b (ties count half)
    if len(a) < 5 or len(b) < 5:
        return np.nan
    r = pd.concat([a, b], ignore_index=True).rank()
    return 1 - (r.iloc[:len(a)].sum() - len(a) * (len(a) + 1) / 2) / (len(a) * len(b))

# two reference indicators (added after seeing the results of the first run)
still_out = pairs[pairs["power_pct"] < 95]                           # nights on which power was not yet restored
dark      = pairs[pairs["power_pct"] < 50]                           # nights with fewer than half of the customers with power
rho_out   = spearman(still_out["ntl_pct"], still_out["power_pct"])
auc_dark  = darker(dark["ntl_pct"], quiet["ntl_pct"])

fig, ax = plt.subplots(figsize=(7.6, 4.3))
x = np.arange(len(ntl_bins))
ax.axhline(100, color=FAINT, lw=1)
ax.vlines(x, ntl_bins["ntl_q25"], ntl_bins["ntl_q75"], color=C_NTL, lw=6, alpha=0.35)
ax.plot(x, ntl_bins["ntl_median"], ls="none", marker="o", ms=9, mfc=C_NTL, mec=SURFACE, mew=1.5)
for xi, (med, n) in enumerate(zip(ntl_bins["ntl_median"], ntl_bins["nights"])):
    if pd.notna(med):
        ax.text(xi + 0.12, med, f"{med:.0f}%", va="center", color=INK, fontsize=10)
    ax.text(xi, -0.115, f"n = {int(n):,}", ha="center", va="top", color=MUTED, fontsize=9, transform=ax.get_xaxis_transform())
ax.set_xticks(x); ax.set_xticklabels(ntl_bins.index); ax.set_xlim(-0.5, len(x) - 0.4); ax.set_ylim(0, 140)
ax.grid(axis="x", visible=False)
ax.set_xlabel("Share of customers with power at overpass (EAGLE-I)", labelpad=32)
ax.set_ylabel("Night lights, percent of normal")
ax.set_title("Night lights on clear nights after landfall, by share of customers with power", pad=22)
ax.text(0, 1.03, f"Median and middle half of county-nights, days 0–14; Spearman over affected counties {rho_nights:.2f}",
        transform=ax.transAxes, color=MUTED, fontsize=9)
fig.savefig(f"{OUT}/step3b_fig1_ntl_vs_power.png"); plt.show()

S = sample.merge(ntl_feat, on=["event", "fips"], how="left")
by = {}
for key, g in S.groupby("event", sort=False):
    p, seen = pairs[pairs["event"] == key], g[g["ntl_nights_7d"] >= 1]
    long72, calm = seen[seen["affected"] & (seen["hours_above"] >= 72)], seen[~seen["affected"]]
    by[key] = {
        "rows": len(g), "base_ok_pct": (g["ntl_base_nights"] >= NTL_MIN_BASE).mean() * 100,
        "clear_7d_pct": (g["ntl_nights_7d"] >= 1).mean() * 100, "median_first_day": g["ntl_first_day"].median(),
        "noise_pct": g["ntl_noise_pct"].median(), "night_pairs": len(p), "rho_nights": spearman(p["ntl_pct"], p["power_pct"]),
        "rho_nights_out": spearman(p.loc[p["power_pct"] < 95, "ntl_pct"], p.loc[p["power_pct"] < 95, "power_pct"]),
        "auc_dark": darker(p.loc[p["power_pct"] < 50, "ntl_pct"], quiet.loc[quiet["event"] == key, "ntl_pct"]),
        "rho_counties": spearman(100 - seen["ntl_min_pct_7d"], seen["hours_above"]),
        "out_72h_seen": len(long72), "out_72h_dim_pct": (long72["ntl_min_pct_7d"] < NTL_LOW_PCT).mean() * 100,
        "not_affected_seen": len(calm), "not_affected_dim_pct": (calm["ntl_min_pct_7d"] < NTL_LOW_PCT).mean() * 100,
    }
ntl_by_event = pd.DataFrame(by).T
ntl_by_event.index.name = "event"
ntl_by_event.to_csv(f"{OUT}/step3b_ntl_by_event.csv")
print(f"Spearman over nights: all {len(pairs):,} nights, {rho_nights:.2f}; only the {len(still_out):,} nights on which power was not yet restored, {rho_out:.2f}")
print(f"Probability that one of the {len(dark):,} nights with under half the customers with power is darker than a night in a county not affected: {auc_dark:.2f}")
display(tidy(ntl_bins))
tidy(ntl_by_event, 2)

**Notes on Figure 1 and the two tables**

- **Figure 1**: from left to right more customers have power; if night lights track outages, the orange dots rise with them. The rightmost group, the counties that were not affected, gives the level of false dimming: its distance from 100% and its `dim_pct` (share of nights below 80%)
- `rho_nights`: within affected counties, the correlation between nightly night lights and the share with power
- `rho_nights_out`, `auc_dark` (added after seeing the results of the first run): on most nights power is already back and night lights are only noise, which pulls `rho_nights` down. `rho_nights_out` uses only the nights with a share with power below 95%; `auc_dark` is the probability that "a night with under half the customers with power" is darker than "a night in a county not affected", where 0.5 means no separation and 1 means complete separation
- `rho_counties`: at county level, whether a deeper drop in night lights within 7 days of landfall goes with a longer outage
- `out_72h_dim_pct` and `not_affected_dim_pct`: the share of counties with outages over 72 hours whose night lights clearly dimmed, and the same share among counties that were not affected. If night lights track outages, the first share is far higher than the second
- `clear_7d_pct` and `median_first_day`: how much cloud got in the way. In a county with no clear night within 7 days, night lights see nothing

In [ ]:
# 12. Figure 2: night lights by days since landfall, within each event
track = nights[nights["clear"] & nights["ntl_pct"].notna()].copy()
track["d"] = np.floor(track["day"]).astype(int)
track["group"] = np.where(track["affected"] & (track["hours_above"] >= 72), "long", np.where(~track["affected"], "quiet", "other"))
GROUPS = {"quiet": ("Not affected", MUTED), "long": ("Outage over 72 hours", C_NTL)}
keys = list(EVENTS.index)
ncol = 3
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(12, 2.6 * nrow), squeeze=False, sharey=True)
for ax, key in zip(axes.flat, keys):
    ax.axhline(100, color=FAINT, lw=1); ax.axvline(0, color=FAINT, lw=1)
    for name, (label, color) in GROUPS.items():
        t = track[(track["event"] == key) & (track["group"] == name)].groupby("d")["ntl_pct"].agg(["median", "size"])
        t = t[(t["size"] >= 5) & (t.index >= -14)]                    # drawn only for days with clear nights in at least 5 counties
        ax.plot(t.index, t["median"], color=color, lw=1.2, marker="o", ms=4, label=label)
    ax.set_title(f"{EVENTS.loc[key, 'name']} {EVENTS.loc[key, 'year']}")
    ax.set_xlim(-14.5, NTL_DAYS[1] + 0.5); ax.set_ylim(0, 140); ax.set_yticks([0, 50, 100])
for ax in axes.flat[len(keys):]:
    ax.set_visible(False)
for ax in axes[-1]:
    ax.set_xlabel("Days since landfall")
for ax in axes[:, 0]:
    ax.set_ylabel("Percent of normal")
axes.flat[0].legend(loc="lower left", fontsize=8.5, handletextpad=0.5)
fig.suptitle("Night lights on clear nights, median across counties", x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.tight_layout()
fig.savefig(f"{OUT}/step3b_fig2_ntl_by_day.png"); plt.show()

## F. Read the flood results; flood features

This cell reads the files that part D exported to Drive. While tasks are still running it waits at most `S1_WAIT_MIN` minutes, then continues with what is there; events that are not finished get empty flood features for now. Once all tasks are done, this cell and the three after it are run again.

Three checks follow the reading:

1. How many counties of each event are covered by radar, and how many have a timely image
2. Whether the area-based `s1_day` of this step agrees with what Step 2c found at the county center points
3. Whether counties with more rain have a larger share flagged as flooded (Figure 3, using `flood_nocrop_pct`, which leaves out wetlands and cropland). No relation at all would mean the flood map is mostly noise

In [ ]:
# 13. Read the flood statistics and compute the flood features
def latest_states():
    try:
        return task_states()
    except Exception:
        return {}

deadline = time.time() + S1_WAIT_MIN * 60
while True:
    missing = [name for k in usable if usable_orbits[k] for name in flood_jobs(k) if not os.path.exists(flood_file(name))]
    states  = latest_states() if missing else {}
    waiting = [name for name in missing if states.get(name, {}).get("state") in ("READY", "RUNNING", "COMPLETED")]
    if not waiting or time.time() >= deadline:
        break
    print(f"{len(waiting)} export tasks have no result yet; waiting 1 minute…")
    time.sleep(60)
for name in missing:
    t = states.get(name, {})
    print(f"{name}: no result yet; task state", t.get("state", "not submitted"), str(t.get("error_message", ""))[:300])

have_flood = [k for k in usable if flood_done(k)]                    # events whose results are complete
parts = []
for k in usable:
    for name in flood_jobs(k):
        if os.path.exists(flood_file(name)):
            d = pd.read_csv(flood_file(name), dtype={"GEOID": str})
            parts.append(d.assign(event=k, fips=d["GEOID"].str.zfill(5))[["event", "fips", "ALAND"] + S1_BANDS])
fl = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["event", "fips", "ALAND"] + S1_BANDS)
fl[["ALAND"] + S1_BANDS] = fl[["ALAND"] + S1_BANDS].astype(float)

land = fl["land_km2"].where(fl["land_km2"] > 0)
fl["s1_cover_pct"] = (land / (fl["ALAND"] / 1e6) * 100).clip(upper=100).fillna(0)
ok = fl["s1_cover_pct"] >= S1_MIN_COVER
fl["s1_day"]           = (fl["day_km2"] / land).where(ok)
fl["s1_timely_pct"]    = (fl["timely_km2"] / land * 100).where(ok)
fl["flood_pct"]        = (fl["flood_km2"] / land * 100).where(ok)
fl["flood_dry_pct"]    = (fl["flood_dry_km2"] / fl["dry_km2"].where(fl["dry_km2"] > 0) * 100).where(ok)
rest = fl["dry_km2"] - fl["crop_km2"]                                 # land that is neither wetland nor cropland
fl["flood_nocrop_pct"] = ((fl["flood_dry_km2"] - fl["flood_crop_km2"]).clip(lower=0) / rest.where(rest > 0) * 100).where(ok)
fl["flood_built_pct"]  = (fl["flood_built_km2"] / fl["built_km2"].where(fl["built_km2"] > 0) * 100).where(ok)
fl["flood_crop_share"] = (fl["flood_crop_km2"] / fl["flood_km2"].where(fl["flood_km2"] > 0) * 100).where(ok)
FLOOD_COLS = ["s1_cover_pct", "s1_day", "s1_timely_pct", "s1_timely", "flood_pct", "flood_dry_pct", "flood_nocrop_pct", "flood_built_pct",
              "flood_crop_share"]
F = sample.merge(fl[["event", "fips"] + [c for c in FLOOD_COLS if c != "s1_timely"]], on=["event", "fips"], how="left")
F["s1_timely"] = F["s1_timely_pct"] >= 50

by = {}
for key, g in F.groupby("event", sort=False):
    t = g[g["s1_timely"] & g["flood_dry_pct"].notna()]
    top = g.loc[g["flood_nocrop_pct"].idxmax()] if g["flood_nocrop_pct"].notna().any() else None
    by[key] = {
        "rows": len(g), "with_result": int(g["s1_cover_pct"].notna().sum()), "covered_90_pct": (g["s1_cover_pct"] >= 90).mean() * 100,
        "timely_pct": g["s1_timely"].mean() * 100, "median_first_day": g["s1_day"].median(),
        "median_flood_dry_pct": g["flood_dry_pct"].median(), "median_flood_nocrop_pct": g["flood_nocrop_pct"].median(),
        "median_crop_share": g["flood_crop_share"].median(), "flooded_1pct_counties": int((g["flood_nocrop_pct"] >= 1).sum()),
        "most_flooded": top["name"] if top is not None else "—", "its_flood_nocrop_pct": top["flood_nocrop_pct"] if top is not None else np.nan,
        "rho_dry_rain": spearman(t["flood_dry_pct"], t["rain_max3d_mm"]), "rho_nocrop_rain": spearman(t["flood_nocrop_pct"], t["rain_max3d_mm"]),
        "rho_nocrop_hours": spearman(t["flood_nocrop_pct"], t["hours_above"]),
    }
flood_by_event = pd.DataFrame.from_dict(by, orient="index")
flood_by_event.index.name = "event"
flood_by_event.to_csv(f"{OUT}/step3b_flood_by_event.csv")

both = F[F["s1_day"].notna() & F["s1_day_step2c"].notna()]
day_gap = (both["s1_day"] - both["s1_day_step2c"]).abs()
print(f"Events with flood results: {len(have_flood)} / {len(usable)}; rows with a result: {int(F['s1_cover_pct'].notna().sum()):,} / {len(F):,}")
print(f"Day of the first image: {len(both):,} rows comparable with Step 2c, median difference {day_gap.median():.2f} days, {int((day_gap > 2).sum())} rows differing by more than 2 days")
tidy(flood_by_event, 2)

In [ ]:
# 14. Figure 3: do counties with more rain have a larger share flagged as flooded; the 15 most flooded counties
T = F[F["s1_timely"] & F["flood_dry_pct"].notna()]
rho_flood_rain  = spearman(T["flood_dry_pct"], T["rain_max3d_mm"])        # the item whose threshold was set beforehand
rho_nocrop_rain = spearman(T["flood_nocrop_pct"], T["rain_max3d_mm"])     # cropland excluded (added after seeing the results)
if len(T):
    fig, ax = plt.subplots(figsize=(6.8, 4.8))
    ax.scatter(T["rain_max3d_mm"], T["flood_nocrop_pct"].clip(lower=0.001), s=14, color=C_POWER, edgecolor=SURFACE, linewidth=0.4, alpha=0.7)
    ax.set_yscale("log")
    top = float(T["flood_nocrop_pct"].max())
    ax.set_ylim(0.0007, (top if top > 0.01 else 0.01) * 6)                 # headroom for the county names
    for i, (_, r) in enumerate(T.nlargest(3, "flood_nocrop_pct").iterrows()):
        ax.annotate(r["name"], (r["rain_max3d_mm"], r["flood_nocrop_pct"]), xytext=(7, 3 + 11 * i), textcoords="offset points", color=INK, fontsize=9)
    ax.set_xlabel("Largest 3-day rainfall in the county (mm)")
    ax.set_ylabel("Land outside wetlands and cropland flagged as flooded (%, log scale)")
    ax.set_title("Flood extent from Sentinel-1 against rainfall", pad=22)
    ax.text(0, 1.03, f"Counties with a timely image (n = {len(T):,}); Spearman {rho_nocrop_rain:.2f}; shares below 0.001% are drawn at 0.001%",
            transform=ax.transAxes, color=MUTED, fontsize=9)
    fig.savefig(f"{OUT}/step3b_fig3_flood_vs_rain.png"); plt.show()
else:
    print("No flood results yet; Figure 3 is not drawn")
print(f"Spearman correlation of flooded share and 3-day rainfall ({len(T):,} counties with a timely image): {rho_flood_rain:.2f} with cropland, {rho_nocrop_rain:.2f} without")
tidy(F.sort_values("flood_nocrop_pct", ascending=False).head(15)[
    ["event", "name", "s1_day", "flood_dry_pct", "flood_nocrop_pct", "flood_built_pct", "flood_crop_share", "rain_max3d_mm", "hours_above"]], 2)

**Notes on Figure 3 and the table above**

- For reference, the known disaster areas are southeastern North Carolina and northeastern South Carolina for Florence; the river valleys of the western North Carolina mountains and the Big Bend coast of Florida for Helene; southwestern and central Florida for Ian; coastal Louisiana for Laura and Ida
- The table is sorted by `flood_nocrop_pct`. A county with a high `flood_dry_pct`, a low `flood_nocrop_pct` and a `flood_crop_share` close to 100% has almost only cropland flagged as flooded, most likely false alarms from the harvest or from flattened crops
- `flood_built_pct` is usually much smaller than `flood_dry_pct`; this is the radar missing floods in built-up areas, not evidence that towns stayed dry

## G. Merge; checklist

The three sets of results are merged into `step3b_features_county_event.csv`: one row per exposed county × event. Step 4 joins it to the Step 3a feature table on `event` and `fips`.

The thresholds of the checklist were set before looking at the results. The three items marked "reference" were added after seeing the results of the first run; they have no preset threshold and are neither passed nor failed. While the flood export tasks are not all finished, the items that depend on them show as "pending".

In [ ]:
# 15. Merge into the Step 3b feature table
feat3b = (sample[["event", "fips", "name"]]
          .merge(hand[["fips"] + HAND_COLS], on="fips", how="left")
          .merge(F[["event", "fips"] + FLOOD_COLS], on=["event", "fips"], how="left")
          .merge(ntl_feat, on=["event", "fips"], how="left"))
feat3b.to_csv(f"{OUT}/step3b_features_county_event.csv", index=False)
print(f"Saved {len(feat3b):,} rows × {feat3b.shape[1]} columns: {len(HAND_COLS)} static, {len(FLOOD_COLS)} flood, {len(NTL_COLS)} night-light")
tidy(feat3b.drop(columns=["event", "fips", "name"]).describe().loc[["count", "min", "50%", "max"]], 2)

In [ ]:
# 16. Step 3b checklist
def verdict(passed):
    if isinstance(passed, str):
        return "reference"
    return "pending" if passed is None else ("pass" if passed else "fail")

base_ok  = (ntl_feat["ntl_base_nights"] >= NTL_MIN_BASE).mean() * 100
clear_7d = (ntl_feat["ntl_nights_7d"] >= 1).mean() * 100
noise    = ntl_feat["ntl_noise_pct"].median()
quiet_md = quiet["ntl_pct"].median()
have_hand = feat3b[HAND_COLS[0]].notna().mean() * 100
done = len(have_flood) == len(usable)                                 # are all flood results in
cover_90 = (F["s1_cover_pct"] >= 90).mean() * 100
gap_md   = day_gap.median()

checks = [
    ("Night lights", "Rows with at least 10 clear nights before landfall (%)", round(base_ok, 1), "≥ 80", base_ok >= 80),
    ("Night lights", "Rows with at least one clear night within 7 days of landfall (%)", round(clear_7d, 1), "≥ 70", clear_7d >= 70),
    ("Night lights", "Median normal night-to-night variation (%)", round(float(noise), 1), "≤ 15", noise <= 15),
    ("Night lights", "Spearman correlation of nightly night lights and share with power, affected counties", round(float(rho_nights), 2), "≥ 0.4", rho_nights >= 0.4),
    ("Night lights", "Median night lights after landfall in counties not affected (normal = 100)", round(float(quiet_md), 1), "90–110", 90 <= quiet_md <= 110),
    ("Night lights", "Spearman correlation of night lights and share with power, nights not yet restored (share below 95%)", round(float(rho_out), 2), "—", "ref"),
    ("Night lights", "Probability that a night with under half the customers with power is darker than a night in a county not affected", round(float(auc_dark), 2), "—", "ref"),
    ("Flood", "Events with flood results", f"{len(have_flood)} / {len(usable)}", f"{len(usable)} / {len(usable)}", True if done else None),
    ("Flood", "Rows with at least 90% of county land covered by radar (%)", round(cover_90, 1), "≥ 85", (cover_90 >= 85) if done else None),
    ("Flood", "Median difference from Step 2c in the day of the first image (days)", round(float(gap_md), 2), "≤ 1", (gap_md <= 1) if done else None),
    ("Flood", "Spearman correlation of flooded share and 3-day rainfall, counties with a timely image", round(float(rho_flood_rain), 2), "≥ 0.2", (rho_flood_rain >= 0.2) if done else None),
    ("Flood", "The same, with cropland left out of the flooded share", round(float(rho_nocrop_rain), 2), "—", "ref"),
    ("Static", "Rows with the share of land close to stream level (%)", round(have_hand, 1), "≥ 99", have_hand >= 99),
]
check_tab = pd.DataFrame(checks, columns=["Part", "Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step3b_checklist.csv", index=False)
check_tab

## Output files

All written to the `us_flood_outage` folder:

- `step3b_features_county_event.csv`, `step3b_checklist.csv`
- `step3b_ntl_by_event.csv`, `step3b_ntl_by_power_bin.csv`, `step3b_ntl_nights.csv.gz`
- `step3b_flood_by_event.csv`, and all the `step3b_flood_<event>.csv` files (events split by state have several)
- `step3b_fig1_ntl_vs_power.png`, `step3b_fig2_ntl_by_day.png`, `step3b_fig3_flood_vs_rain.png`